# Pricing Analysis Pipeline

## Overview
This notebook builds a **pricing analysis data pipeline** that ingests raw order data (2023–2025), customer demographics, and product catalogs, then produces summary tables for downstream reporting and visualization.

## Data Sources
All source files are stored in Unity Catalog Volume: `/Volumes/Pricing_analysis/data/all_datasets/`

| File | Description |
| --- | --- |
| `Orders_2023.csv` | Order transactions for 2023 |
| `Orders_2024.csv` | Order transactions for 2024 |
| `Orders_2025.csv` | Order transactions for 2025 |
| `customers.csv` | Customer master data (Region, Join Date) |
| `products.csv` | Product catalog (Name, Category, Price, Base Cost) |

## Output Tables
| Table | Purpose |
| --- | --- |
| `Pricing_analysis.data.orders_master` | Denormalized fact table with all orders enriched with customer and product attributes |
| `Pricing_analysis.data.monthly_performance` | Aggregated monthly KPIs by category and region |
| `Pricing_analysis.data.product_profitability` | Product-level profitability metrics |

## Key Metrics Computed
* **Gross Profit** = Revenue − COGS
* **Gross Margin %** = (Revenue − COGS) / Revenue × 100
* **Average Selling Price** = Revenue / Quantity
* **Markup %** = (Avg Selling Price − Base Cost) / Base Cost × 100

## Step 1: Create Master Orders Table

This step unions all yearly order CSVs (2023–2025) into a single dataset and enriches each order with:
* **Customer data** — Region and Join Date (via `CustomerID` join)
* **Product data** — Product Name, Category, List Price, and Base Cost (via `ProductID` join)

Derived columns include `GrossProfit`, `GrossMarginPct`, `AvgSellingPrice`, and `MarkupPct` to support profitability analysis downstream.

In [0]:
%sql
CREATE OR REPLACE TABLE Pricing_analysis.data.orders_master AS
WITH orders_union AS (
  SELECT * FROM read_files('/Volumes/Pricing_analysis/data/all_datasets/Orders_2023.csv', format => 'csv', header => true, inferSchema => true)
  UNION ALL
  SELECT * FROM read_files('/Volumes/Pricing_analysis/data/all_datasets/Orders_2024.csv', format => 'csv', header => true, inferSchema => true)
  UNION ALL
  SELECT * FROM read_files('/Volumes/Pricing_analysis/data/all_datasets/Orders_2025.csv', format => 'csv', header => true, inferSchema => true)
),
customers AS (
  SELECT * FROM read_files('/Volumes/Pricing_analysis/data/all_datasets/customers.csv', format => 'csv', header => true, inferSchema => true)
),
products AS (
  SELECT * FROM read_files('/Volumes/Pricing_analysis/data/all_datasets/products.csv', format => 'csv', header => true, inferSchema => true)
)
SELECT
  o.OrderID,
  o.CustomerID,
  c.Region,
  CAST(c.CustomerJoinDate AS DATE) AS CustomerJoinDate,
  o.ProductID,
  p.ProductName,
  p.ProductCategory,
  p.Price         AS ListPrice,
  p.Base_Cost     AS BaseCost,
  CAST(o.OrderDate AS DATE) AS OrderDate,
  YEAR(CAST(o.OrderDate AS DATE))  AS OrderYear,
  MONTH(CAST(o.OrderDate AS DATE)) AS OrderMonth,
  DATE_TRUNC('month', CAST(o.OrderDate AS DATE)) AS OrderMonthDate,
  o.Quantity,
  o.Revenue,
  o.COGS,
  o.Revenue - o.COGS                                              AS GrossProfit,
  ROUND((o.Revenue - o.COGS) / NULLIF(o.Revenue, 0) * 100, 2)   AS GrossMarginPct,
  ROUND(o.Revenue / NULLIF(o.Quantity, 0), 2)                    AS AvgSellingPrice,
  ROUND((o.Revenue / NULLIF(o.Quantity, 0) - p.Base_Cost) / NULLIF(p.Base_Cost, 0) * 100, 2) AS MarkupPct
FROM orders_union o
LEFT JOIN customers c ON o.CustomerID = c.CustomerID
LEFT JOIN products  p ON CAST(o.ProductID AS INT) = p.ProductID;


## Step 2: Monthly Performance Summary

Aggregates the master orders table into a monthly time-series view, grouped by **Product Category** and **Region**. This powers trend charts and period-over-period comparisons.

**Metrics produced:**
* Order Count, Total Units Sold
* Total Revenue, Total COGS, Total Gross Profit
* Gross Margin % and Average Selling Price

In [0]:
%sql
CREATE OR REPLACE TABLE Pricing_analysis.data.monthly_performance AS
SELECT
  OrderMonthDate,
  OrderYear,
  OrderMonth,
  ProductCategory,
  Region,
  COUNT(DISTINCT OrderID)     AS OrderCount,
  SUM(Quantity)               AS TotalUnits,
  ROUND(SUM(Revenue), 2)      AS TotalRevenue,
  ROUND(SUM(COGS), 2)         AS TotalCOGS,
  ROUND(SUM(GrossProfit), 2)  AS TotalGrossProfit,
  ROUND(SUM(GrossProfit) / NULLIF(SUM(Revenue), 0) * 100, 2) AS GrossMarginPct,
  ROUND(AVG(AvgSellingPrice), 2) AS AvgSellingPrice
FROM Pricing_analysis.data.orders_master
GROUP BY OrderMonthDate, OrderYear, OrderMonth, ProductCategory, Region;


## Step 3: Product Profitability Analysis

Builds a product-level summary showing each product’s pricing structure and realized profitability.

**Key columns:**
* `UnitMargin` / `MarginPct` / `MarkupPct` — list-price-based margins
* `TotalOrders` / `TotalUnitsSold` — volume indicators
* `ActualGrossMarginPct` — realized margin after discounts and volume effects

In [0]:
CREATE OR REPLACE TABLE Pricing_analysis.data.product_profitability AS
SELECT
  ProductID,
  ProductName,
  ProductCategory,
  ListPrice,
  BaseCost,
  ROUND(ListPrice - BaseCost, 2)                                AS UnitMargin,
  ROUND((ListPrice - BaseCost) / NULLIF(ListPrice, 0) * 100, 2) AS MarginPct,
  ROUND((ListPrice - BaseCost) / NULLIF(BaseCost, 0) * 100, 2)  AS MarkupPct,
  COUNT(DISTINCT OrderID)        AS TotalOrders,
  SUM(Quantity)                  AS TotalUnitsSold,
  ROUND(SUM(Revenue), 2)         AS TotalRevenue,
  ROUND(SUM(GrossProfit), 2)     AS TotalGrossProfit,
  ROUND(SUM(GrossProfit) / NULLIF(SUM(Revenue), 0) * 100, 2) AS ActualGrossMarginPct
FROM Pricing_analysis.data.orders_master
GROUP BY ProductID, ProductName, ProductCategory, ListPrice, BaseCost;

## Step 4: Validation

Quick row-count check to confirm all tables were created successfully and contain the expected data volume.

In [0]:
SELECT 'orders_master' AS table_name, COUNT(*) AS row_count FROM Pricing_analysis.data.orders_master
UNION ALL
SELECT 'monthly_performance', COUNT(*) FROM Pricing_analysis.data.monthly_performance
UNION ALL
SELECT 'product_profitability', COUNT(*) FROM Pricing_analysis.data.product_profitability;